# 03 — Run canonical hedging

Runs the canonical **test-holdout hedging analysis** from the persisted calibration workbooks. By default it runs both WLS (`lambda_TS = 0`) and PWLS (`lambda_TS = 1`); PWLS remains the final specification.

By default the notebook uses every available snapshot (`START_DT = END_DT = None`) and writes both `excel files/canonical/hedging_wls.xlsx` and `excel files/canonical/hedging_pwls.xlsx`. The hedging engine applies the same optional date window before constructing forward snapshot pairs.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib.common import configure_notebook_display, locate_notebook_paths
PATHS = locate_notebook_paths(NOTEBOOK_DIR)
ROOT = PATHS.project_root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

configure_notebook_display(max_columns=200, width=200)
print("Project root:", ROOT)

from _lib.stage3_workflow import (
    CANONICAL_HEDGING_REPORT_SUBDIR,
    build_hedging_configs,
    run_hedging_reporting,
    run_hedging_suite,
)


In [ ]:
START_DT = None
END_DT = None
RUN_WLS = True
RUN_PWLS = True

configs = build_hedging_configs(
    ROOT,
    run_wls=RUN_WLS,
    run_pwls=RUN_PWLS,
    start_dt=START_DT,
    end_dt=END_DT,
)
for key, cfg in configs.items():
    print(key.upper(), "calibration ->", cfg.calibration_xlsx)
    print(key.upper(), "hedging ->", cfg.output_xlsx)
    if key == "pwls":
        print("Canonical final workbook: hedging_pwls.xlsx")


In [ ]:
for key, cfg in configs.items():
    if not cfg.calibration_xlsx.exists():
        raise FileNotFoundError(f"Run 01_run_calibration.ipynb first; missing {key}: {cfg.calibration_xlsx}")
print("Calibration pre-flight passed.")


In [ ]:
hedging_workbooks = run_hedging_suite(configs)
hedging_workbooks


In [ ]:
HEDGE_REPORT_ROOT = NOTEBOOK_DIR / CANONICAL_HEDGING_REPORT_SUBDIR
hedging_reports = run_hedging_reporting(
    hedging_workbooks,
    HEDGE_REPORT_ROOT,
    q=5,
    make_figures=True,
)
for key, report in hedging_reports.items():
    print(key.upper(), "representative model:", report["representative_model"])
    display(report["tables"]["headline_test"])
